# **MetaPassAV:** Preparing the Data for 16-Avatars

_Cleaning Raw Data_

### Libraries and Constants

In [2]:
import pandas as pd

from scripts.AvatarFeatureEngineer import *
from scripts.AvatarVisualizer import *
from scripts.AvatarExperiment import *

---

## Data Loading

In [ ]:
BASEPATH = "./"
DATA_PATH = f"{BASEPATH}/data"

In [4]:
df = pd.read_csv(f"{DATA_PATH}/raw_data.csv")
df.shape

(4121, 14)

In [5]:
df.head()

,UserID,ImageName,Order,StartTime,EndTime,TotalTimeSpent,Direction,AngleChange,HeadX,HeadY,HeadZ,DistanceFromStart,DistanceFromPrevious,gender
0,User-1,image5,1,58.83619,62.83619,4.0,Left Forward,87.62966,-23.38911,1.843688,-29.12778,2.524077,2.524077,1
1,User-1,image5,1,62.83619,66.83619,4.0,Stable,87.62966,-23.38911,1.843688,-29.12778,2.524077,2.524077,1
2,User-1,image5,1,66.83619,70.83619,4.0,Stable,87.62966,-23.38911,1.843688,-29.12778,2.524077,2.524077,1
3,User-1,image5,1,70.83619,74.83619,4.0,Stable,87.62966,-23.38911,1.843688,-29.12778,2.524077,2.524077,1
4,User-1,image5,1,74.83619,78.83619,4.0,Stable,87.62966,-23.38911,1.843688,-29.12778,2.524077,2.524077,1


In [6]:
df.UserID.value_counts()

UserID
User-9     402
User-10    386
User-4     377
User-2     358
User-1     343
User-8     341
User-7     335
User-3     306
User-5     293
User-6     257
User-16    147
User-15    141
User-12    128
User-13    112
User-14    111
User-11     84
Name: count, dtype: int64

---

## Init Data Understanding

In [6]:
df.isna().sum()

UserID                  0
ImageName               0
Order                   0
StartTime               0
EndTime                 0
TotalTimeSpent          0
Direction               0
AngleChange             0
HeadX                   0
HeadY                   0
HeadZ                   0
DistanceFromStart       0
DistanceFromPrevious    0
gender                  0
dtype: int64

In [7]:
df.describe()

,Order,StartTime,EndTime,TotalTimeSpent,AngleChange,HeadX,HeadY,HeadZ,DistanceFromStart,DistanceFromPrevious,gender
count,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000
mean,5.498423,106.304653,109.760611,3.455959,78.216646,-22.420256,1.642889,-29.074526,19.097179,15.860359,0.567095
std,3.441192,71.405322,71.431353,1.084562,50.319742,17.151395,0.962432,13.708566,11.108399,14.843654,0.495538
min,1.000000,4.700531,8.700531,0.005690,0.894374,-47.948830,0.932381,-48.349340,2.524077,0.000633,0.000000
25%,3.000000,54.022500,56.965270,3.659550,39.633980,-38.311310,1.155294,-36.112720,14.625320,5.622814,0.000000
50%,5.000000,92.186360,95.548130,4.000000,69.652840,-26.515380,1.496542,-31.562250,17.462840,11.904090,1.000000
75%,8.000000,141.063600,144.261000,4.000000,116.302600,-5.796408,1.848222,-27.423200,21.647990,22.378590,1.000000
max,19.000000,479.429100,481.206800,4.000000,179.916100,11.803720,10.677120,47.376390,82.331120,97.861520,1.000000


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4121 entries, 0 to 4120
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   UserID                4121 non-null   object 
 1   ImageName             4121 non-null   object 
 2   Order                 4121 non-null   int64  
 3   StartTime             4121 non-null   float64
 4   EndTime               4121 non-null   float64
 5   TotalTimeSpent        4121 non-null   float64
 6   Direction             4121 non-null   object 
 7   AngleChange           4121 non-null   float64
 8   HeadX                 4121 non-null   float64
 9   HeadY                 4121 non-null   float64
 10  HeadZ                 4121 non-null   float64
 11  DistanceFromStart     4121 non-null   float64
 12  DistanceFromPrevious  4121 non-null   float64
 13  gender                4121 non-null   int64  
dtypes: float64(9), int64(2), object(3)
memory usage: 450.9+ KB


---

In [ ]:
df.UserID.nunique(), df.UserID.value_counts()

In [ ]:
df.ImageName.nunique(), df.ImageName.value_counts()

#### Important Features:

```
['TotalTimeSpent', 
 'HeadX', 
 'HeadY', 
 'HeadZ', 
 'DistanceFromPrevious', 
 'DistanceFromStart', 
 'DistanceSoFar', 
 '3DAngularChange', 
 'AngularShift', 
 'LinVelocity', 
 'LinAcceleration', 
 'Jerk', 
 'Snap', 
 'AngularShiftRate', 
 'AngularShiftJerkRate', 
 'gender'
]
```

---

In [ ]:
df_bkp = df.copy()

In [ ]:
df = df_bkp.copy()

## Data Preparation

### Assigning Session ID

In [ ]:
def assign_global_session_ids(
    df: pd.DataFrame,
    user_col: str = "UserID",
    time_col: str = "StartTime",
    session_col: str = "SessionID",
    prefix: str = "Session"
) -> pd.DataFrame:
    """
    Assigns globally unique session IDs based on StartTime reset per user.

    A new session is created whenever StartTime decreases or stays the same
    compared to the previous row *within the same user*.

    Session IDs are global (never reused across users) and string-labeled:
    e.g., Session_1, Session_2, ...
    """

    df = df.copy()

    # Detect session breaks per user
    session_break = (
        df[time_col]
          .diff()
          .le(0)
          .fillna(True)
    )

    # Global session init index holder
    global_session_index = session_break[session_break == True].index.tolist()
    global_session_index.append(df.shape[0])
    
    # Aggregating global session IDs
    global_session_ids = []
    i = 1
    lastSi = 0
    for thisSi in global_session_index:
        this_session_ids = [f'{prefix}_{i}' for j in range(thisSi-lastSi)]
        global_session_ids.extend(this_session_ids)
        lastSi = thisSi
        i += 1

    # Assign string-based global session IDs
    df[session_col] = global_session_ids
    
    return df

In [ ]:
df = assign_global_session_ids(df)

---

## Experiments

### Feature Engineering

In [ ]:
# Create experiment object
exp = AvatarExperiment(df)

### Feature Analysis

In [ ]:
# Get feature-enriched dataframe
df = exp.get_features()
df.columns

In [ ]:
df.SessionID.nunique(), df.SessionID.value_counts()

In [ ]:
df.head()

In [ ]:
df[['Direction', 'AngleChange', 'HeadX', 'HeadY', 'HeadZ',
       'DistanceFromStart', 'DistanceFromPrevious','SessionID',
       'DistanceSoFar', 'bearing_azimuth', 'bearing_elevation',
       '3DAngularChange', 'AngularShift', 'LinVelocity', 'LinAcceleration',
       'Jerk', 'Snap', 'AngularShiftRate', 'AngularShiftJerkRate']]#.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.Direction.value_counts()

---

### Visual Analysis

#### User Specific Plots

##### User-3

In [ ]:
# female
df[df.UserID == "User-3"].SessionID.value_counts()

In [ ]:
# Get all trijactories
for s in [f"Session_{tsid}" for tsid in range(23, 34)]:
    viz = exp.get_visualizer(
        user_id="User-3",
        session_id=s
    )
    
    viz.plot_trajectory_xz()

In [ ]:
# Get all bearing angles
for s in [f"Session_{tsid}" for tsid in range(23, 34)]:
    viz = exp.get_visualizer(
        user_id="User-3",
        session_id=s
    )
    
    viz.plot_bearing()

##### User-9

In [ ]:
# male
df[df.UserID == "User-9"].SessionID.value_counts()

In [ ]:
# Get visualizer directly (no manual plumbing)
viz = exp.get_visualizer(
    user_id="User-9",
    session_id="Session_93"
)

In [ ]:
# Run exploratory plots
viz.plot_trajectory_xz()
viz.plot_distance_progress()
viz.plot_velocity_acceleration()
viz.plot_jerk_snap()
viz.plot_direction_sequence()
viz.plot_feature_histogram("Snap")

viz.plot_dwell_time()
viz.plot_bearing()
viz.plot_angular_dynamics()

---  
---  

## Data Cleaning

In [ ]:
df.info()

### Direct Feature Removal

In [ ]:
df.columns

In [ ]:
COL_TO_DROP = ['Order', 'gender']
df.drop(columns=COL_TO_DROP, inplace=True)

In [ ]:
df.shape

### Analytics Based Dropping

#### Sneak Peek

In [ ]:
df.Direction.value_counts()

In [ ]:
df[df.columns[:10]].describe()

In [ ]:
df[df.columns[10:20]].describe()

In [ ]:
df[df.columns[20:]].describe()

---

In [ ]:
df = df[['SessionID', 'UserID', 'StartTime', 'EndTime', 'TotalTimeSpent', 
         'ImageName', 'HeadX', 'HeadY', 'HeadZ', 
         'DistanceFromStart', 'DistanceFromPrevious', 'DistanceSoFar', 
         'Direction', 'AngleChange', 'bearing_azimuth', 'bearing_elevation',
         '3DAngularChange', 'AngularShift',
         'LinVelocity', 'LinAcceleration', 'Jerk', 'Snap', 
         'AngularShiftRate', 'AngularShiftJerkRate']]

df.tail()

#### Feature Removal

In [ ]:
MUST_KEEP_COLS = ['DistanceFromStart',
                  'DistanceFromPrevious',
                  'DistanceSoFar',
                  # possible min-max norm
                  'AngularShift',
                  'LinVelocity',
                  'AngularShiftRate',
                  'TotalTimeSpent',
                  'AngleChange',
                  # categorical --> OHE
                  'Direction',
                  # use as it is
                  'bearing_elevation'] 

MAY_KEEP_COLS = ['AngularShiftJerkRate', 
                 'Jerk',
                 # categorical --> OHE
                 'ImageName'] 

## not for model training
KEEP_FOR_DATA_MANAGEMENT = ['UserID', 'SessionID', 'StartTime']


## less impact, redundant, or misleading columns
DROP_COLS = ['EndTime',
             'HeadX', 'HeadY', 'HeadZ',
             # 'Order', 'gender', # ALREADY DROPPED
             # misleading/low importance
             'Snap', 'LinAcceleration', 'bearing_azimuth', '3DAngularChange']

---

In [ ]:
df[MUST_KEEP_COLS].describe()

In [ ]:
df[MAY_KEEP_COLS].describe()

In [ ]:
df[DROP_COLS].describe()

---

In [ ]:
df.drop(columns=DROP_COLS, inplace=True)
df.shape

### Catagorical Feature Encoding

#### One-Hot Encoding

In [ ]:
# for Direction
direction_subDF = pd.get_dummies(df["Direction"], prefix="Dir")
df = pd.concat([df, direction_subDF], axis=1)
df.shape, df.columns

In [ ]:
# for ImageName
image_subDF = pd.get_dummies(df["ImageName"])
df = pd.concat([df, image_subDF], axis=1)
df.shape, df.columns

In [ ]:
df.drop(columns=['Direction', 'ImageName'], inplace=True)
df.shape

## Exporting Enriched Heater Dataset

In [ ]:
df.info()

In [ ]:
df = df[['SessionID', 'StartTime', 'TotalTimeSpent',
         'DistanceFromStart', 'DistanceFromPrevious', 'DistanceSoFar',
         'AngleChange', 'bearing_elevation', 'AngularShift', 'LinVelocity',
         'Jerk', 'AngularShiftRate', 'AngularShiftJerkRate', 
         # catag. -> OHE
         'Dir_B', 'Dir_F', 'Dir_L', 'Dir_LB', 'Dir_LF', 'Dir_R', 'Dir_RB', 
         'Dir_RF', 'Dir_S',
         'image1', 'image2', 'image3', 'image4', 'image5', 'image6', 
         'image7', 'image8', 
         # label
         'UserID']]

In [ ]:
df.head()

In [53]:
df.shape

(4121, 31)

In [54]:
df.to_csv(f"{DATA_PATH}/AvatarBeh_FeatureRich.csv", index=False)

---  
---  